# 09.6 — Extending the pipeline

You've traced the whole pipeline — data, architecture, training, losses, curriculum, output, deployment. This final notebook closes the loop: **how do you add something new?** A new encoder architecture, a new loss component, a new curriculum regime, a new decoding target. The answer is the same for all four, and it's the design principle the whole codebase is built on: **extend by registering, not by editing core code.** The pipeline exposes registries and config-driven dispatch, so a new component plugs in from *outside* — the open/closed principle in practice. This notebook is that extension guide, and the curriculum's send-off.

This notebook covers:

* The extension philosophy: register, don't modify (open/closed).
* Adding an architecture — the `@register_encoder` decorator, live.
* Adding a loss component, a curriculum regime, and a target task.
* Where each extension point lives.

**Prerequisites:** [04.1 architecture dispatcher](../04_architecture/04.1_architecture_string_dispatcher.ipynb), [06.1 loss overview](../06_loss_orchestration/06.1_multi_task_losses_overview.ipynb), [07.6 the curriculum walkthrough](../07_dynamic_curriculum/07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb).

## Section 1 — What MATLAB does

In MATLAB, adding an architecture means *editing* a central `switch` statement — `cgg_constructNetworkArchitecture.m` grows a new `case`:

```matlab
switch ModelName
    case 'GRU'      % existing
        layers = buildGRU(cfg);
    case 'MyNewNet' % <- you edit this file to add your case
        layers = buildMyNewNet(cfg);
end
```

Every extension touches the shared file, so the core code grows with each addition and two people extending it collide. The port inverts this: the core defines a *registry*, and a new architecture *registers itself* from its own module — the core file never changes. Same for losses, curricula, and targets. Extension is additive, not invasive.

## Section 2 — The Python concepts you need

### 2.1 — Register, don't modify (the open/closed principle)

The design goal: the pipeline should be **open for extension** (you can add new architectures/losses/…) but **closed for modification** (you don't edit the core to do it). A *registry* achieves this — a lookup table that maps a name to a builder. The core looks up whatever's registered; your new component registers itself under a new name from its own file. Nobody edits a shared `switch`. This is why [04.1](../04_architecture/04.1_architecture_string_dispatcher.ipynb)'s string-dispatch pattern recurs everywhere: the encoder registry, the classifier registry, the schedule library, the loss aggregator — all are registries, all extensible the same way.

### 2.2 — Add an architecture: `@register_encoder`

In [ ]:
import neural_data_decoding.models       # importing triggers the built-in registrations
from neural_data_decoding.models import registry

print("encoders registered out of the box:", registry.list_encoders())
print("classifiers:", registry.list_classifiers())

In [ ]:
import torch.nn as nn

# Add YOUR OWN encoder — just decorate a builder function. No core file edited.
if "MyToyEncoder" not in registry.list_encoders():        # (guard so the cell re-runs)
    @registry.register_encoder("MyToyEncoder")
    def _build_my_toy(cfg):
        return nn.Linear(cfg.get("in_dim", 4), cfg.get("out_dim", 8))

# It's now a first-class citizen — build it by name like any built-in:
model = registry.build_encoder("MyToyEncoder", {"in_dim": 4, "out_dim": 8})
print("built 'MyToyEncoder':", model)
print("now in the registry?", "MyToyEncoder" in registry.list_encoders())
print("→ one decorator added an architecture the whole pipeline can use — no core code touched.")

`@registry.register_encoder("MyToyEncoder")` adds a builder to the encoder registry. From that moment, `build_encoder("MyToyEncoder", cfg)` works exactly like a built-in — the config's `model_name` can select it, a sweep can vary to it ([09.4](09.4_parameter_sweeps.ipynb)), the folder hierarchy names it ([08.1](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb)). The registration is an *import side-effect*: put the decorated builder in a module, import that module, and the architecture exists. To ship it, you'd add the module to the `models` package's imports so it registers at startup. There's also a parallel flag-bundle registry (`architecture_registry.py`, `ArchitectureSpec` / `resolve_architecture`) for architectures that are just *parameter variations* of existing builders — the [04.1](../04_architecture/04.1_architecture_string_dispatcher.ipynb) spec approach.

### 2.3 — Add a loss component

A new loss term plugs into the multi-objective aggregator ([06.1](../06_loss_orchestration/06.1_multi_task_losses_overview.ipynb), [06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)). The pattern: write the loss *kernel* (in `training/losses/`), then thread it through `aggregate_normalized_losses` as a new keyword argument with its own weight key:

In [ ]:
import inspect
from neural_data_decoding.training.losses.multi_objective import aggregate_normalized_losses

# The aggregator's parameters ARE the extension surface — each loss component is a kwarg:
params = list(inspect.signature(aggregate_normalized_losses).parameters)
print("aggregate_normalized_losses parameters:")
for p in params:
    print(f"   {p}")
print()
print("→ a new component = a new *_loss kwarg + a key in `weights` + its own EMA prior.")
print("  The EMA normalization (06.4) then balances it against the others automatically.")

To add, say, a temporal-smoothness loss: write `temporal_smoothness_loss(...)` in `training/losses/`, add a `temporal_smoothness_loss=` parameter to the aggregator, give it a `"temporal_smoothness"` weight key and an EMA prior in `LossPriors`, and route it into the decoder or classifier sub-total ([06.11](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)). The EMA prior normalization ([06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)) then scales it against the existing components automatically — you don't have to hand-tune its magnitude, only its *weight* (relative importance). The aggregator is the single place losses compose, so a new one is a localized change, not a scatter of edits.

### 2.4 — Add a curriculum regime (no code at all)

In [ ]:
from neural_data_decoding.training.schedules.library import slugify_regime

# A new curriculum is a YAML FILE, not code. The regime name maps to a filename:
for regime in ["My Custom Warmup", "Aggressive KL Annealing"]:
    print(f"'{regime}' → configs/schedule/{slugify_regime(regime)}.yaml")
print()
print("→ write that YAML (weights/freeze/augmentation waypoints, 07.2) and set")
print("  cfg.dynamic_parameter_set = 'My Custom Warmup'. No Python changes needed.")

A new curriculum regime ([07.6](../07_dynamic_curriculum/07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb)) needs *no code* — it's a YAML file. `slugify_regime` maps the regime's name to a filename in `configs/schedule/`; you author that file with `weights`/`freeze`/`augmentation` waypoint blocks ([07.2](../07_dynamic_curriculum/07.2_piecewise_linear_schedules.ipynb)), then point `cfg.dynamic_parameter_set` at the regime name. `load_curriculum_by_name` finds and builds it. This is config-as-extension at its purest: the curriculum *is* data, so adding one is authoring data, not editing logic.

### 2.5 — Add a target task

In [ ]:
# The decoding target is a plain config field — no registry, just cfg-driven dispatch.
# The pipeline reads cfg.target to pick which labels to decode.
example_targets = ["Dimension", "Outcome"]     # the shipped targets
print("cfg.target selects what to decode:", example_targets)
print()
print("A new target (say 'ReactionTime') means:")
print("  1. teach the dataset to produce that label column (data/*.py)")
print("  2. set cfg.target = 'ReactionTime'")
print("→ no registry — the target is config-driven dispatch inside the dataset + run-namer.")

A new *decoding target* — what the model predicts — is the lightest extension: `cfg.target` is a plain config field (`Dimension`, `Outcome`, …) that the dataset and the folder-namer read. To add one, teach the dataset to produce that label and set `cfg.target`. There's no registry because the target isn't *behavior* to swap — it's a *column of data* to select. It's even already a sweep dimension ([09.4](09.4_parameter_sweeps.ipynb)). The four extension points span the spectrum: a decorator (architecture), a kwarg (loss), a YAML file (curriculum), a config field (target) — from most code to none, but all *additive*.

## Section 3 — The `neural_data_decoding` implementation

`register_encoder` / `build_encoder` — the registry that makes architectures extensible:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/registry.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def register_encoder"))
for k in range(i, i + 4):
    print(f"{k + 1:4} | {src[k]}")
j0 = next(j for j, line in enumerate(src) if line.startswith("def build_encoder"))
for k in range(j0, j0 + 4):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `register_encoder(name)` returns a decorator that stores the builder under `name`; `build_encoder(name, cfg)` looks it up and calls it. The registry is a plain dict — the simplest possible extension mechanism.
* A duplicate registration *raises* (`ValueError`) — you can't silently shadow an existing architecture, so a name collision is caught at import, not at runtime.
* `build_encoder` on an unknown name raises with the *list of valid options* — a helpful error that tells you what you can use (and reveals typos).
* The built-in encoders register as import side-effects of `neural_data_decoding.models` (`encoder.py`, `conv_encoder.py`); classifiers likewise. Your extension does the same from your module.
* The same shape appears across the codebase — `register_classifier`/`build_classifier`, the schedule library, the loss aggregator — so learning one registry teaches them all ([04.1](../04_architecture/04.1_architecture_string_dispatcher.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — register a second encoder

Register another toy encoder under a new name and confirm both yours and the built-ins coexist in the registry.

In [ ]:
# Reveal:
if "AnotherEncoder" not in registry.list_encoders():
    @registry.register_encoder("AnotherEncoder")
    def _build_another(cfg):
        return nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8))

encoders = registry.list_encoders()
print("total encoders now:", len(encoders))
print("both custom encoders present?",
      "MyToyEncoder" in encoders and "AnotherEncoder" in encoders)
print("built-ins still there?", "GRU" in encoders and "Convolutional" in encoders)
print("→ extensions and built-ins coexist — the registry just grew.")

### Exercise 4.2 — a collision is caught

Show that registering a *duplicate* name raises, so you can't accidentally shadow an existing architecture.

In [ ]:
# Reveal:
try:
    @registry.register_encoder("GRU")        # already taken!
    def _shadow(cfg):
        return nn.Identity()
    print("no error (unexpected)")
except ValueError as e:
    print("registering 'GRU' again →", type(e).__name__, ":", str(e)[:60])
print("→ duplicate names are rejected at import — collisions fail loud, not silent.")

## Section 5 — Common errors

### The new architecture isn't found at runtime

Its module was never imported, so the registration side-effect never ran (§2.2). Ensure the module is imported at startup (add it to the `models` package's imports), or the registry never learns about it.

### `ValueError: already registered`

Two builders claimed the same name (§Exercise 4.2). Pick a unique name — the registry deliberately refuses to shadow, so a collision is a real conflict to resolve, not to override.

### A new loss has no effect

Check all three: the kernel is written, the aggregator has the new kwarg *and* it's summed into a sub-total ([06.11](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)), and the `weights` dict has a non-zero key for it ([06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)). Miss any and the term is silently dropped (a `NaN`/`0`/absent weight, [06.11 §5](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)).

### A new curriculum regime raises `FileNotFoundError`

`slugify_regime(name)` must map to an existing `configs/schedule/*.yaml` (§2.4). Confirm the filename matches the slug of your regime name exactly.

### Editing core files to extend

If you find yourself adding a `case`/`if` to a shared dispatcher, stop — there's almost certainly a registry or config field for it (§2.1). Editing core is the MATLAB pattern the port was designed to *avoid*.

## Section 6 — Further reading

- [`src/neural_data_decoding/models/registry.py`](../../src/neural_data_decoding/models/registry.py) — the encoder/classifier registries.
- [`src/neural_data_decoding/training/losses/multi_objective.py`](../../src/neural_data_decoding/training/losses/multi_objective.py) — the loss extension surface.
- [`src/neural_data_decoding/training/schedules/library.py`](../../src/neural_data_decoding/training/schedules/library.py) — `slugify_regime`, the curriculum extension point.

---

**🎓 Curriculum complete.** You began at [00.1](../00_orientation/00.1_welcome.ipynb) as a MATLAB programmer meeting Python, and you've now traced this entire neural-decoding pipeline end to end: the [orientation](../00_orientation/) and [Python](../01_python_for_matlab_users/) foundations; [NumPy/PyTorch](../02_numpy_and_pytorch_basics/) and the [data pipeline](../03_data_pipeline/); the [architecture](../04_architecture/), [training loop](../05_training_loop/), [loss orchestration](../06_loss_orchestration/), and [dynamic curriculum](../07_dynamic_curriculum/); the [output & analysis](../08_output_and_analysis/) bridge back to MATLAB; and [production deployment](../09_production_deployment/) on a cluster. Along the way, the recurring lessons — *PyTorch's defaults are not MATLAB's*, *read the code, not the label*, *verify parity empirically*, *register, don't modify* — are the ones that transfer to any deep-learning codebase you meet next. You're ready to extend this pipeline, and to read the next one. Go build something.